# Example 9.5.1 / 2: Write with Quotes and Overwrite

Companion notebook for **9.5.1 CSV Files** in the AMPL lab Documentation, section "Write with Quotes and Overwrite". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Write with Quotes and Overwrite

`quote=double` puts string values in double quotes when writing. `overwrite` replaces the
file, header included; without it an `OUT` write keeps the file's existing header:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param market {PROD} >= 0;

table Quoted OUT "amplcsv" "quoted.csv" "quote=double" "overwrite":
    PROD -> [product], market;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}

In [5]:
# run.py
ampl.eval("load amplcsv.dll;")
ampl.write_table("Quoted")
print(open("quoted.csv").read())

"product","market"
"bands",6000
"coils",4000
"plate",3500



### Pure AMPL: data.dat and model.run

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param market := bands 6000  coils 4000  plate 3500;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
load amplcsv.dll;
write table Quoted;
display market;

market [*] :=
bands  6000
coils  4000
plate  3500
;

